In [25]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [26]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    
    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [27]:
def calculate_sr(state: BatsmanState):
    sr = (state["runs"] / state["balls"]) * 100
    state["sr"] = sr
    return {"sr": sr}

def calculate_bpb(state: BatsmanState):
    bpb = state["balls"] / (state["fours"] + state["sixes"])
    state["bpb"] = bpb
    return {"bpb": bpb}

def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (
        ((state["fours"] * 4) + (state["sixes"] * 6))
        / state["runs"]
    ) * 100

    state["boundary_percent"] = boundary_percent
    return {"boundary_percent": boundary_percent}

def summary(state: BatsmanState):
    summary = f"""
Strike Rate = {state['sr']}
Balls per boundary = {state['bpb']}
Boundary percent = {state['boundary_percent']}
"""

    state["summary"] = summary
    return state

In [28]:
graph = StateGraph(BatsmanState)
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("summary", summary)


# Parallel branches
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

# Wait for all branches before running summary
graph.add_edge(
    [
        "calculate_sr",
        "calculate_bpb",
        "calculate_boundary_percent",
    ],
    "summary",
)

graph.add_edge("summary", END)

workflow = graph.compile()


In [29]:
initial_state = {
    "runs": 100,   
    "balls": 60,
    "fours": 10,
    "sixes": 5,
    "sr": 0.0,
    "bpb": 0.0,
    "boundary_percent": 0.0,
    "summary": "",
}

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 60,
 'fours': 10,
 'sixes': 5,
 'sr': 166.66666666666669,
 'bpb': 4.0,
 'boundary_percent': 70.0,
 'summary': '\nStrike Rate = 166.66666666666669\nBalls per boundary = 4.0\nBoundary percent = 70.0\n'}